# 🎬 Pipeline de Edición de Video — Reels / TikTok
**Canal:** Duvan — IA aplicada a Marketing y Ecommerce

### Fases:
1. ✂️ Cortar silencios
2. 📝 Transcribir (AssemblyAI)
3. 💬 Subtítulos animados WebM
4. 🎨 Keyword pills overlay
5. 🎥 Composite final

**Ejecuta cada celda con `Shift+Enter` en orden.**

In [ ]:
# ── CELDA 1: SETUP (tarda ~1 min) ──────────────────────────────────────
import subprocess, sys

subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'assemblyai', 'pillow', 'gdown'], check=True)

import requests
FONT_SEMIBOLD  = '/tmp/Inter-SemiBold.ttf'
FONT_BOLD      = '/tmp/Inter-Bold.ttf'
FONT_EXTRABOLD = '/tmp/Inter-ExtraBold.ttf'
try:
    base = 'https://github.com/rsms/inter/raw/master/docs/font-files/'
    for fname, dst in [('Inter-SemiBold.ttf',  FONT_SEMIBOLD),
                        ('Inter-Bold.ttf',       FONT_BOLD),
                        ('Inter-ExtraBold.ttf',  FONT_EXTRABOLD)]:
        r = requests.get(base + fname, timeout=15)
        if r.status_code == 200:
            open(dst, 'wb').write(r.content)
    print('✅ Fuente Inter descargada')
except Exception as e:
    FONT_SEMIBOLD = FONT_BOLD = FONT_EXTRABOLD = None
    print(f'⚠️  Inter no disponible, usando fuente del sistema')

print('✅ Setup completo')

In [ ]:
# ── CELDA 2: DESCARGAR VIDEO DESDE GOOGLE DRIVE ─────────────────────────
import gdown, os
from pathlib import Path

# ID del archivo en Google Drive
DRIVE_FILE_ID = '1YUw2aeO0UcshwF76h8mhHSlBRKQda45s'
VIDEO_INPUT   = '/content/video_input.mp4'

if not Path(VIDEO_INPUT).exists():
    print('Descargando video desde Google Drive...')
    gdown.download(id=DRIVE_FILE_ID, output=VIDEO_INPUT, quiet=False)
else:
    print('Video ya descargado')

size_mb = Path(VIDEO_INPUT).stat().st_size / 1e6
print(f'✅ {VIDEO_INPUT} ({size_mb:.1f} MB)')

In [ ]:
# ── CELDA 3: CONFIGURACIÓN ──────────────────────────────────────────────

# API key de AssemblyAI (assemblyai.com → gratis)
ASSEMBLYAI_API_KEY = 'd12c2fe0480a4e049332291c685d2d57'

# Hook box — None = sin hook (solo subtítulos + keyword pills)
HOOK_TEXT = None

SILENCE_MIN_S    = 0.5    # segundos mínimos de silencio a cortar
SILENCE_NOISE_DB = -35    # umbral dB
WORDS_PER_CUE    = 2      # palabras por cue de subtítulos
LANGUAGE         = 'es'

OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Configuración lista')
print(f'   Hook: {"desactivado" if HOOK_TEXT is None else HOOK_TEXT[:40]}')
print(f'   Video: {VIDEO_INPUT}')

In [ ]:
# ── CELDA 4: FUNCIONES (no editar) ──────────────────────────────────────
import json, re, subprocess
from pathlib import Path

def get_duration(p):
    r = subprocess.run(['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
                        '-of', 'csv=p=0', str(p)], capture_output=True, text=True, check=True)
    return float(r.stdout.strip())

def silence_cut(input_path, output_path, min_silence_s=0.5, noise_db=-35):
    print(f'Detectando silencios...')
    r = subprocess.run(
        ['ffmpeg', '-i', str(input_path),
         '-af', f'silencedetect=noise={noise_db}dB:d={min_silence_s}',
         '-f', 'null', '-'], capture_output=True, text=True)

    starts, ends = [], []
    for line in r.stderr.split('\n'):
        m = re.search(r'silence_start: ([\d.]+)', line)
        if m: starts.append(float(m.group(1)))
        m = re.search(r'silence_end: ([\d.]+)', line)
        if m: ends.append(float(m.group(1)))

    duration = get_duration(input_path)
    silences = list(zip(starts, ends[:len(starts)]))
    print(f'  {len(silences)} silencios — duración fuente: {duration:.1f}s')

    keeps, t = [], 0.0
    for ss, se in sorted(silences):
        if ss - t > 0.15: keeps.append({'start': round(t,3), 'end': round(ss,3)})
        t = se
    if duration - t > 0.15: keeps.append({'start': round(t,3), 'end': round(duration,3)})

    out_dur = sum(k['end']-k['start'] for k in keeps)
    print(f'  {len(keeps)} segmentos → {out_dur:.1f}s de output')

    segs = []
    for i, k in enumerate(keeps):
        sp = f'/tmp/seg_{i:03d}.mp4'
        subprocess.run(['ffmpeg', '-y', '-ss', str(k['start']), '-to', str(k['end']),
                        '-i', str(input_path), '-c:v', 'libx264', '-preset', 'ultrafast',
                        '-crf', '18', '-c:a', 'aac', '-b:a', '192k', sp],
                       capture_output=True, check=True)
        segs.append(sp)
        print(f'  segmento {i+1}/{len(keeps)}', end='\r')

    cl = '/tmp/concat_list.txt'
    Path(cl).write_text('\n'.join(f"file '{s}'" for s in segs))
    subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', cl,
                    '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
                    '-c:a', 'aac', '-b:a', '192k', str(output_path)],
                   capture_output=True, check=True)

    final = get_duration(output_path)
    print(f'\n✅ Base: {output_path}  ({duration:.1f}s → {final:.1f}s, -{duration-final:.1f}s)')
    return keeps, final


def transcribe_assemblyai(video_path, api_key, language='es'):
    import assemblyai as aai
    aai.settings.api_key = api_key
    audio = '/tmp/audio_mono.wav'
    subprocess.run(['ffmpeg', '-y', '-i', str(video_path),
                    '-ac', '1', '-ar', '16000', '-sample_fmt', 's16', audio],
                   capture_output=True, check=True)
    print('Transcribiendo con AssemblyAI...')
    t = aai.Transcriber().transcribe(
        audio, aai.TranscriptionConfig(language_code=language, speech_model=aai.SpeechModel.best))
    if t.status == aai.TranscriptStatus.error:
        raise RuntimeError(f'AssemblyAI: {t.error}')
    words = [{'text': w.text, 'start': w.start, 'end': w.end, 'type': 'word'}
             for w in (t.words or [])]
    print(f'✅ {len(words)} palabras transcritas')
    for w in words[:5]: print(f"   {w['start']/1000:.2f}s  '{w['text']}'")
    return {'words': words, 'text': t.text}


def build_subtitle_cues(words, words_per_cue=2):
    wl = [{'text': w['text'], 'start_s': w['start']/1000, 'end_s': w['end']/1000}
          for w in words if w.get('type','word') == 'word']
    cues = []
    for i in range(0, len(wl), words_per_cue):
        g = wl[i:i+words_per_cue]
        ns = wl[i+words_per_cue]['start_s'] if i+words_per_cue < len(wl) else g[-1]['end_s']+0.5
        cues.append({'start_s': g[0]['start_s'], 'duration_s': max(ns-g[0]['start_s'],0.2),
                     'words': [w['text'] for w in g]})
    return cues


KEYWORD_PILLS = {
    'claude':'Claude', 'claude code':'Claude Code',
    'chatgpt':'ChatGPT', 'chat gpt':'ChatGPT', 'gpt':'GPT',
    'meta ads':'Meta ADs', 'facebook ads':'Facebook Ads', 'fb ads':'FB Ads',
    'instagram':'Instagram', 'tiktok':'TikTok',
    'agente ia':'Agente IA', 'agente':'Agente IA',
    'automatizacion':'Automatización', 'automatización':'Automatización',
    'ugc':'UGC', 'ecommerce':'Ecommerce', 'whatsapp':'WhatsApp',
    'notion':'Notion', 'make':'Make.com', 'n8n':'n8n',
    'openai':'OpenAI', 'gemini':'Gemini', 'anthropic':'Anthropic',
    'blender':'Blender', 'cursor':'Cursor', 'windsurf':'Windsurf',
}

def detect_keyword_events(words):
    events = []
    for i, w in enumerate(words):
        wl = w['text'].lower().strip('.,!?¡¿:;"\'')
        nl = words[i+1]['text'].lower().strip('.,!?¡¿:;"\'')\
             if i+1 < len(words) else ''
        bg = wl + ' ' + nl
        matched = next((d for kw,d in KEYWORD_PILLS.items()
                        if (' ' in kw and kw in bg) or (' ' not in kw and kw == wl)), None)
        if matched:
            events.append({'start_s': w['start']/1000, 'text': matched, 'duration_s': 2.0})
    deduped, last = [], {}
    for ev in sorted(events, key=lambda x: x['start_s']):
        if ev['start_s'] - last.get(ev['text'], -99) > 4.0:
            deduped.append(ev)
            last[ev['text']] = ev['start_s']
    return deduped

print('✅ Funciones cargadas')

In [ ]:
# ── CELDA 5: RENDERIZADORES PIL (no editar) ─────────────────────────────
from PIL import Image, ImageDraw, ImageFont

def load_font(path, size):
    for p in [path,
              '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
              '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf']:
        if p:
            try: return ImageFont.truetype(p, size)
            except: pass
    return ImageFont.load_default()

def alpha_at(t, start, dur, fade=0.12):
    end = start + dur
    if t < start or t > end: return 0.0
    fade = min(fade, dur * 0.3)
    if t - start < fade: return (t - start) / fade
    if end - t   < fade: return (end - t)   / fade
    return 1.0

def ffmpeg_webm_proc(output_path, canvas_w=1080, canvas_h=1920, fps=30):
    return subprocess.Popen([
        'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
        '-s', f'{canvas_w}x{canvas_h}', '-pix_fmt', 'rgba',
        '-r', str(fps), '-i', 'pipe:0',
        '-c:v', 'libvpx-vp9', '-pix_fmt', 'yuva420p',
        '-b:v', '0', '-crf', '32', '-auto-alt-ref', '0',
        str(output_path)
    ], stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)


def render_subtitles_webm(cues, output_path, video_duration,
                           fps=30, canvas_w=1080, canvas_h=1920,
                           font_size=28, bottom=385, gap=5):
    font = load_font(FONT_SEMIBOLD, font_size)
    total = int(video_duration * fps) + 1
    proc  = ffmpeg_webm_proc(output_path, canvas_w, canvas_h, fps)

    for fi in range(total):
        t   = fi / fps
        img = Image.new('RGBA', (canvas_w, canvas_h), (0,0,0,0))
        draw = ImageDraw.Draw(img)
        ac  = next((c for c in cues if c['start_s'] <= t < c['start_s']+c['duration_s']), None)

        if ac:
            a     = alpha_at(t, ac['start_s'], ac['duration_s'])
            words = ac['words']
            sizes = []
            for w in words:
                bb = draw.textbbox((0,0), w, font=font)
                sizes.append((bb[2]-bb[0]+20, bb[3]-bb[1]+8))
            tw   = sum(s[0] for s in sizes) + gap*(len(words)-1)
            mh   = max(s[1] for s in sizes)
            x    = (canvas_w - tw) // 2
            y    = canvas_h - bottom - mh
            for wi, (word, (ww, _)) in enumerate(zip(words, sizes)):
                active = wi == len(words)-1
                bg = (17,17,17,int(240*a))   if active else (255,255,255,int(235*a))
                fg = (255,255,255,int(255*a)) if active else (17,17,17,int(255*a))
                bb = draw.textbbox((0,0), word, font=font)
                th = bb[3]-bb[1]
                draw.rounded_rectangle([x,y,x+ww,y+mh], radius=5, fill=bg)
                draw.text((x+10, y+(mh-th)//2), word, fill=fg, font=font)
                x += ww + gap

        proc.stdin.write(img.tobytes())
        if fi % (fps*10) == 0:
            print(f'  subs {fi}/{total} ({t:.0f}s)', end='\r')

    proc.stdin.close(); proc.wait()
    if proc.returncode != 0: raise RuntimeError('ffmpeg falló — subtítulos')
    print(f'\n✅ Subtítulos: {output_path}')


def render_overlay_webm(hook_text, keyword_events, output_path, video_duration,
                         fps=30, canvas_w=1080, canvas_h=1920):
    """Overlay con hook opcional y keyword pills. hook_text=None → solo pills."""
    font_h = load_font(FONT_BOLD,      32)
    font_k = load_font(FONT_EXTRABOLD, 38)
    total  = int(video_duration * fps) + 1
    proc   = ffmpeg_webm_proc(output_path, canvas_w, canvas_h, fps)

    for fi in range(total):
        t   = fi / fps
        img = Image.new('RGBA', (canvas_w, canvas_h), (0,0,0,0))
        draw = ImageDraw.Draw(img)

        # ── Hook (solo si hay texto) ───────────────────────────────────
        if hook_text:
            ha = alpha_at(t, 0.0, 5.0, fade=0.4)
            if ha > 0:
                lines = hook_text.replace('\\n', '\n').split('\n')
                lsizes = [draw.textbbox((0,0), l, font=font_h) for l in lines]
                lsizes = [(b[2]-b[0], b[3]-b[1]) for b in lsizes]
                mxw  = max(s[0] for s in lsizes)
                lh   = max(s[1] for s in lsizes)
                tot  = len(lines)*lh + (len(lines)-1)*8
                bw, bh = mxw+48, tot+32
                bx = (canvas_w-bw)//2
                ai = int(255*ha)
                draw.rounded_rectangle([bx,110,bx+bw,110+bh], radius=12,
                                        fill=(255,255,255,int(242*ha)))
                ty = 110+16
                for line,(lw,lhh) in zip(lines,lsizes):
                    draw.text((bx+(bw-lw)//2, ty), line, fill=(17,17,17,ai), font=font_h)
                    ty += lhh+8

        # ── Keyword pills ──────────────────────────────────────────────
        for ev in keyword_events:
            a = alpha_at(t, ev['start_s'], ev['duration_s'], fade=0.25)
            if a <= 0: continue
            ai = int(255*a)
            bb = draw.textbbox((0,0), ev['text'], font=font_k)
            tw, th = bb[2]-bb[0], bb[3]-bb[1]
            pw, ph = tw+40, th+20
            px = (canvas_w-pw)//2
            py = int(canvas_h*0.42)-ph//2
            draw.rounded_rectangle([px,py,px+pw,py+ph], radius=10, fill=(17,17,17,ai))
            draw.text((px+20,py+10), ev['text'], fill=(255,255,255,ai), font=font_k)

        proc.stdin.write(img.tobytes())
        if fi % (fps*10) == 0:
            print(f'  overlay {fi}/{total} ({t:.0f}s)', end='\r')

    proc.stdin.close(); proc.wait()
    if proc.returncode != 0: raise RuntimeError('ffmpeg falló — overlay')
    print(f'\n✅ Overlay: {output_path}')


def verify_alpha(path):
    r = subprocess.run(['ffprobe', '-v', 'quiet', '-show_entries', 'stream=pix_fmt',
                        '-of', 'default=noprint_wrappers=1', str(path)],
                       capture_output=True, text=True)
    ok = 'yuva420p' in r.stdout
    print(f'  alpha: {"✅ yuva420p" if ok else "⚠️ " + r.stdout.strip()}')
    return ok

print('✅ Renderizadores listos')

In [ ]:
# ── FASE 1: CORTAR SILENCIOS ────────────────────────────────────────────
from pathlib import Path
from IPython.display import Image as IPImage, display

BASE_VIDEO = Path(OUTPUT_DIR) / 'base_video.mp4'
keeps, base_duration = silence_cut(VIDEO_INPUT, BASE_VIDEO, SILENCE_MIN_S, SILENCE_NOISE_DB)

# Frame de verificación
chk = str(Path(OUTPUT_DIR) / 'check_fase1.jpg')
subprocess.run(['ffmpeg', '-y', '-ss', '3', '-i', str(BASE_VIDEO),
                '-vframes', '1', '-q:v', '2', chk], capture_output=True)
print('\n📸 Frame segundo 3:')
display(IPImage(chk, width=280))

In [ ]:
# ── FASE 2: TRANSCRIBIR ─────────────────────────────────────────────────
import json
transcript_path = Path(OUTPUT_DIR) / 'transcript.json'
transcript = transcribe_assemblyai(BASE_VIDEO, ASSEMBLYAI_API_KEY, LANGUAGE)
transcript_path.write_text(json.dumps(transcript, ensure_ascii=False, indent=2))
print(f'\n💾 {transcript_path}  ({len(transcript["words"])} palabras)')

In [ ]:
# ── FASE 3: SUBTÍTULOS WebM ─────────────────────────────────────────────
subs_webm = Path(OUTPUT_DIR) / 'subtitles.webm'
cues = build_subtitle_cues(transcript['words'], WORDS_PER_CUE)
print(f'{len(cues)} cues — primeros 3: {[c["words"] for c in cues[:3]]}')

render_subtitles_webm(cues, subs_webm, base_duration)
verify_alpha(subs_webm)

In [ ]:
# ── FASE 4: OVERLAY (keyword pills) ─────────────────────────────────────
overlay_webm   = Path(OUTPUT_DIR) / 'overlay.webm'
keyword_events = detect_keyword_events(transcript['words'])

print(f'Keywords detectadas: {len(keyword_events)}')
for ev in keyword_events:
    print(f"  {ev['start_s']:.2f}s → '{ev['text']}'")

# Renderizar solo si hay algo que mostrar (hook o keywords)
if HOOK_TEXT or keyword_events:
    render_overlay_webm(HOOK_TEXT, keyword_events, overlay_webm, base_duration)
    verify_alpha(overlay_webm)
    SKIP_OVERLAY = False
else:
    print('\nℹ️  Sin hook ni keywords detectadas — overlay omitido')
    SKIP_OVERLAY = True

In [ ]:
# ── FASE 5: COMPOSITE FINAL ─────────────────────────────────────────────
final_output = Path(OUTPUT_DIR) / 'final_video.mp4'

if SKIP_OVERLAY:
    # Solo base + subtítulos
    cmd = [
        'ffmpeg', '-y',
        '-i', str(BASE_VIDEO),
        '-vcodec', 'libvpx-vp9', '-i', str(subs_webm),
        '-filter_complex', '[0:v][1:v]overlay=x=0:y=0:format=auto[vout]',
        '-map', '[vout]', '-map', '0:a',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18', '-pix_fmt', 'yuv420p',
        '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart',
        str(final_output)
    ]
else:
    # Base + subtítulos + overlay
    cmd = [
        'ffmpeg', '-y',
        '-i', str(BASE_VIDEO),
        '-vcodec', 'libvpx-vp9', '-i', str(subs_webm),
        '-vcodec', 'libvpx-vp9', '-i', str(overlay_webm),
        '-filter_complex',
        '[0:v][1:v]overlay=x=0:y=0:format=auto[v1];[v1][2:v]overlay=x=0:y=0:format=auto[vout]',
        '-map', '[vout]', '-map', '0:a',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18', '-pix_fmt', 'yuv420p',
        '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart',
        str(final_output)
    ]

print('Compositando...'); res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print('ERROR:'); print(res.stderr[-3000:])
else:
    mb = Path(final_output).stat().st_size / 1e6
    print(f'✅ {final_output}  ({mb:.1f} MB)')

    from IPython.display import display as idisplay
    for ts, label in [(3,'inicio'), (15,'medio'), (30,'subs')]:
        chk = str(Path(OUTPUT_DIR) / f'check_{label}.jpg')
        subprocess.run(['ffmpeg', '-y', '-ss', str(ts), '-i', str(final_output),
                        '-vframes', '1', '-q:v', '2', chk], capture_output=True)
        if Path(chk).exists():
            print(f'\n📸 Segundo {ts}:')
            idisplay(IPImage(chk, width=280))

In [ ]:
# ── DESCARGAR ───────────────────────────────────────────────────────────
from google.colab import files
files.download(str(final_output))
print(f'✅ Descargando {Path(final_output).name}')